# 02 — Masking

**Purpose:** Apply point-source and cluster masks to the full-sky HEALPix CIB and tSZ maps.

This notebook covers both masking steps described in §2 of the paper:

1. **Point-source masking** — uses `get_point_source_mask_in_healpix` to identify pixels
   with flux above 2 mJy and sets them to zero. Note: the original `preprocessing.ipynb`
   used `astropy.stats.sigma_clip` instead; see `docs/paper_code_inconsistencies.md` for
   discussion.

2. **Cluster masking** — uses `get_apodised_mdpl2_cluster_mask` with the halo catalogue
   from `01_halo_catalogue.ipynb` to build apodised circular masks around clusters with
   M500c ≥ 3×10¹⁴ M☉, with radii of 3θ₅₀₀c–5θ₅₀₀c and a minimum of 10'. Masked pixels
   are inpainted with Gaussian random values matching the map mean and standard deviation.

**Inputs:**
- Full-sky CIB map: `data/agora_len_mag_cibmap_act_150ghz.fits`
- Full-sky tSZ map: `data/agora_ltszNG_bahamas80_bnd_unb_1.0e+12_1.0e+18_lensed.fits`
- Halo catalogue: `data/halo_catalogue/halo_catalogue_m500gt3e14.npy` (from `01_halo_catalogue.ipynb`)

**Outputs:**
- Masked CIB map: `data/cib_150_masked.fits`
- Masked tSZ map: `data/tsz_150_masked.fits`

**Key module functions:**
- `foregrounds_diffusion.get_cluster_source_mask_for_agora.get_point_source_mask_in_healpix`
- `foregrounds_diffusion.get_cluster_source_mask_for_agora.get_apodised_mdpl2_cluster_mask`

**Paper reference:** §2 (point-source and cluster masking descriptions).

In [13]:
!pip install -e ~/cmb_foregrounds_diffusion/

Obtaining file:///home/apb86/cmb_foregrounds_diffusion
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for foregrounds_diffusion (pyproject.toml) ... done
  Created wheel for foregrounds_diffusion: filename=foregrounds_diffusion-0.1.0-0.editable-py3-none-any.whl size=4647 sha256=e190a416d77c4557e82f649a99a6219a59e7676a5c01d7f8994ec3c358da21c3
  Stored in directory: /tmp/pip-ephem-wheel-cache-kums71ji/wheels/ad/05/47/d622ea03cc2c607997f65b7643dab9c4f27fa5f37d82097115
Successfully built foregrounds_diffusion
  Attempting uninstall: foregrounds_diffusion
    Found existing installation: foregrounds_diffusion 0.1.0
    Uninstalling foregrounds_diffusion-0.1.0:
      Successfully uninstalled foregrounds_diffusion-0.1.0

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip insta

In [14]:
import numpy as np
import healpy as hp

from pathlib import Path

print("Imported numpy and healpy")

from foregrounds_diffusion.get_cluster_source_mask_for_agora import (
    get_point_source_mask_in_healpix,
    get_apodised_mdpl2_cluster_mask,
    get_mdpl2_conversion_factors_K_to_MjyperSr,
)

from foregrounds_diffusion.preprocessing import get_peak_masks

# Anchor to the project root regardless of where the notebook is launched from
PROJECT_ROOT = Path("/home/apb86/cmb_foregrounds_diffusion")
HPC_WORK = Path("/home/apb86/rds/hpc-work")

CIB_FITS = PROJECT_ROOT / "data" / "mdpl2_len_mag_cibmap_act_150_uk.fits"
TSZ_FITS = PROJECT_ROOT / "data" / "mdpl2_ltszNG_bahamas80_rot_sum_4_176_bnd_unb_1.0e+12_1.0e+18_v103021_lmax24000_nside8192_interp1.0_method1_1_lensed_map.fits"
HALO_CAT = HPC_WORK / "halo_catalogue" / "halo_catalogue_m500gt3e14.npy.npz"

FREQ              = 150.      # GHz
PTSRC_THRESH_MJY  = 2.0       # mJy threshold at 150 GHz
M500C_THRESHOLD   = 3e14      # M_sun


Imported numpy and healpy


In [15]:
import os
print(os.getcwd())

/home/apb86/cmb_foregrounds_diffusion/docs/tutorials


In [16]:
cib_map = hp.read_map(CIB_FITS)
tsz_map = hp.read_map(TSZ_FITS)
nside   = hp.get_nside(cib_map)

print(f"NSIDE = {nside},  Npix = {hp.nside2npix(nside):,}")
print(f"CIB — min: {cib_map.min():.4f}  max: {cib_map.max():.4f}  std: {cib_map.std():.4f}")
print(f"tSZ — min: {tsz_map.min():.4f}  max: {tsz_map.max():.4f}  std: {tsz_map.std():.4f}")


NSIDE = 8192,  Npix = 805,306,368
CIB — min: -10.6890  max: 74069.4766  std: 21.8938
tSZ — min: -0.0000  max: 0.0003  std: 0.0000


Convert tSZ map to $\mu\;\text{K}$ and convert both maps to NSIDE=2048.

In [17]:
cib_map_2048 = hp.pixelfunc.ud_grade(cib_map, nside_out=2048)
tsz_map_2048 = hp.pixelfunc.ud_grade(tsz_map, nside_out=2048) * 1e6
hp.fitsfunc.write_map("~/cmb_foregrounds_diffusion/data/cib_map_uK_2048", cib_map_2048, overwrite=True)
hp.fitsfunc.write_map("~/cmb_foregrounds_diffusion/data/tsz_map_uK_2048", tsz_map_2048, overwrite=True)

setting the output map dtype to [dtype('float32')]
setting the output map dtype to [dtype('float32')]


In [19]:
# Identify pixels significantly above the global mean
mean_val = np.mean(cib_map_2048)
std_val = np.std(cib_map_2048)
sigma_thresh = 5.0
bright_pixels = np.where(cib_map_2048 > mean_val + sigma_thresh * std_val)[0]
print(f"Bright pixels above {sigma_thresh}σ: {len(bright_pixels):,} ({100*len(bright_pixels)/len(cib_map_2048):.2f}%)")

# Punch holes around each bright pixel
nside_map = hp.get_nside(cib_map_2048)
npix = hp.nside2npix(nside_map)
ptsrc_mask = np.ones(npix, dtype=np.float32)
mask_radius_rad = np.radians(3. / 60.)  # 3 arcmin radius

for pix in bright_pixels:
    ivec = hp.pix2vec(nside_map, pix)
    disc = hp.query_disc(nside_map, ivec, mask_radius_rad)
    ptsrc_mask[disc] = 0.

print(f"Pixels masked: {(ptsrc_mask == 0).sum():,} ({100*(ptsrc_mask == 0).mean():.2f}%)")

cib_ptsrc = cib_map_2048 * ptsrc_mask
tsz_ptsrc = tsz_map_2048 * ptsrc_mask


Bright pixels above 5.0σ: 19,337 (0.04%)
Pixels masked: 168,146 (0.33%)


In [20]:
!pip install colossus


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [21]:
# Build apodised cluster mask for halos with M500c >= 3e14 M_sun
cluster_mask = get_apodised_mdpl2_cluster_mask(
    nside=2048,
    halo_cat_fname=HALO_CAT,
    m500c_threshold=M500C_THRESHOLD,
    howmanythetaforclusters=3,   # mask radius = 3 × theta_500c
    apodise=True,
)
print(f"Cluster mask: sky fraction retained = {cluster_mask.mean():.3f}")

print(f"cluster_mask — min: {cluster_mask.min():.4e}  max: {cluster_mask.max():.4e}  mean: {cluster_mask.mean():.4f}")
print(f"cib_ptsrc — min: {cib_ptsrc.min():.4e}  max: {cib_ptsrc.max():.4e}")
print(f"tsz_ptsrc — min: {tsz_ptsrc.min():.4e}  max: {tsz_ptsrc.max():.4e}")

import gc

CIB_FITS_OUT = PROJECT_ROOT / "data" / "cib_masked.fits"
TSZ_FITS_OUT = PROJECT_ROOT / "data" / "tsz_masked.fits"

masked_idx = np.where(cluster_mask < 0.5)[0]
unmasked_idx = np.where(cluster_mask >= 0.5)[0]

rng = np.random.default_rng(seed=0)

# CIB
cib_masked = cib_ptsrc * cluster_mask
del cib_ptsrc
cib_fill = rng.normal(loc=cib_masked[unmasked_idx].mean(),
                      scale=cib_masked[unmasked_idx].std(),
                      size=len(masked_idx))
cib_masked[masked_idx] = cib_fill
del cib_fill
hp.write_map(CIB_FITS_OUT, cib_masked.astype(np.float32), overwrite=True)
del cib_masked
gc.collect()

# tSZ
tsz_masked = tsz_ptsrc * cluster_mask
del tsz_ptsrc
tsz_fill = rng.normal(loc=tsz_masked[unmasked_idx].mean(),
                      scale=tsz_masked[unmasked_idx].std(),
                      size=len(masked_idx))
tsz_masked[masked_idx] = tsz_fill
del tsz_fill
hp.write_map(TSZ_FITS_OUT, tsz_masked.astype(np.float32), overwrite=True)
del tsz_masked, cluster_mask, masked_idx, unmasked_idx
gc.collect()

print("Saved masked maps.")

	Total clusters to mask: 6686
0
1000
2000
3000
4000
5000
6000
0
5000
Starting apodisation
Cluster mask: sky fraction retained = 0.987
cluster_mask — min: 0.0000e+00  max: 1.0000e+00  mean: 0.9871
cib_ptsrc — min: 0.0000e+00  max: 5.2465e+01
tsz_ptsrc — min: 0.0000e+00  max: 2.7058e+02


setting the output map dtype to [dtype('float32')]
setting the output map dtype to [dtype('float32')]


Saved masked maps.


In [22]:
import resource
mem = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
print(f"Peak memory usage: {mem / 1024:.1f} MB")

Peak memory usage: 16980.6 MB
